# 08. Hyperparameter Tuning Using Grid Search and Random Search

## 📚 Learning Objectives

By completing this notebook, you will:
- Perform hyperparameter tuning
- Use Grid Search
- Use Random Search
- Optimize model parameters
- Compare search strategies

## 🔗 Where this fits

**Builds on:** Course 04 (AIAT 114) — Unit 5, lesson 01 "Grid Search and Random Search" — the same searches, now compared for cost as well as score.

**Used later in:** Course 12 (AIAT 126) — Unit 3, lesson 02, which tunes the graduation-project model.

---

This notebook covers practical activities from **Course 05, Unit 4**:
- Hyperparameter tuning using techniques like Grid Search and Random Search

---

## Introduction

**Hyperparameter tuning** optimizes model performance by systematically searching for the best parameter values using Grid Search or Random Search techniques.


## The Story

**BEFORE**: You can train models but don't know how to optimize their performance.

**AFTER**: You'll learn hyperparameter tuning: grid search, random search, and finding the best model parameters!

**Why this matters**: Hyperparameter Tuning Using Grid Search and Random Search is essential for building complete, professional data science solutions!

---

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Model, param grid
- sklearn

**Outputs:** What you'll see when you run the cells

- Best params
- CV results
- Printed summary

---

In [1]:
# WHAT: Import search tools and recap what hyperparameters are.
# WHY: Hyperparameters are set BEFORE training; finding good ones is a search problem with real compute cost.

# Imports
import time
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     cross_val_score, train_test_split)
from scipy.stats import randint

print("✅ Libraries imported!")
print("\nHyperparameter Tuning: Grid Search and Random Search")
print("=" * 60)
print("""
Hyperparameters are the knobs YOU set before training (n_estimators,
max_depth, ...), unlike parameters the model learns (coefficients, splits).

Remember Example 07's dead end: model selection cost =
  (parameter combinations) x (CV folds) x (per-fit time).
This notebook runs that search for real, twice:
  - Grid search: try EVERY combination
  - Random search: sample a fixed number of combinations

We tune on real clinical data: the Wisconsin Diagnostic Breast Cancer set,
569 tumour biopsies described by 30 measurements taken from digitised images
of a fine-needle aspirate, each labelled malignant or benign.
""")

✅ Libraries imported!

Hyperparameter Tuning: Grid Search and Random Search

Hyperparameters are the knobs YOU set before training (n_estimators,
max_depth, ...), unlike parameters the model learns (coefficients, splits).

Remember Example 07's dead end: model selection cost =
  (parameter combinations) x (CV folds) x (per-fit time).
This notebook runs that search for real, twice:
  - Grid search: try EVERY combination
  - Random search: sample a fixed number of combinations

We tune on real clinical data: the Wisconsin Diagnostic Breast Cancer set,
569 tumour biopsies described by 30 measurements taken from digitised images
of a fine-needle aspirate, each labelled malignant or benign.



## Part 1: Baseline — Default Hyperparameters

Before tuning anything, measure what the default model gives us.

In [2]:
# WHAT: Load the real breast-cancer diagnostic data and score a default random forest with 3-fold CV.
# WHY: The untuned baseline is the yardstick - tuning only matters if it beats this number.

print("Part 1: Baseline with default hyperparameters")
print("-" * 60)

# Real data: Street, Wolberg & Mangasarian (1993), University of Wisconsin.
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Dataset: {X.shape[0]} real biopsies x {X.shape[1]} measurements")
print(f"Classes: {dict(zip(cancer.target_names, np.bincount(y)))}")
print(f"Train {len(X_train)} / test {len(X_test)}")

start = time.time()
baseline_scores = cross_val_score(
    RandomForestClassifier(random_state=42), X_train, y_train, cv=3)
baseline_time = time.time() - start
baseline_score = baseline_scores.mean()
print(f"\nDefault RandomForest, 3-fold CV accuracy: {baseline_score:.4f}")
print(f"Per-fold scores: {np.round(baseline_scores, 4)}")
print(f"Time for the baseline evaluation: {baseline_time:.2f}s")
print("\n💡 A strong default baseline is normal on this dataset - which makes it a")
print("   good honesty test: watch how little tuning actually buys.")

Part 1: Baseline with default hyperparameters
------------------------------------------------------------
Dataset: 569 real biopsies x 30 measurements
Classes: {np.str_('malignant'): np.int64(212), np.str_('benign'): np.int64(357)}
Train 426 / test 143

Default RandomForest, 3-fold CV accuracy: 0.9624
Per-fold scores: [0.9577 0.9577 0.9718]
Time for the baseline evaluation: 0.17s

💡 A strong default baseline is normal on this dataset - which makes it a
   good honesty test: watch how little tuning actually buys.


## Part 2: Grid Search — Try Every Combination

`GridSearchCV` cross-validates **every** combination in the grid and refits
the best one on the full training set.

In [3]:
# WHAT: Run GridSearchCV over 8 parameter combinations and report the best model's CV and test accuracy.
# WHY: Grid search is exhaustive within its grid - reliable, but its cost multiplies with every added knob.

print("Part 2: Grid Search")
print("-" * 60)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, None],
    'min_samples_split': [2, 5],
}
n_combos = 2 * 2 * 2
n_folds = 3
print(f"Grid: {param_grid}")
print(f"Cost: {n_combos} combinations x {n_folds} folds (+1 refit) = {n_combos * n_folds + 1} fits")

start = time.time()
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid, cv=n_folds, n_jobs=-1)
grid.fit(X_train, y_train)
grid_time = time.time() - start

print(f"\nGrid search finished in {grid_time:.2f}s")
print(f"Best parameters: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}  (baseline was {baseline_score:.4f})")
grid_test = grid.score(X_test, y_test)
print(f"Held-out test accuracy of the tuned model: {grid_test:.4f}")

Part 2: Grid Search
------------------------------------------------------------
Grid: {'n_estimators': [50, 100], 'max_depth': [5, None], 'min_samples_split': [2, 5]}
Cost: 8 combinations x 3 folds (+1 refit) = 25 fits



Grid search finished in 3.19s
Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 50}
Best CV accuracy: 0.9624  (baseline was 0.9624)
Held-out test accuracy of the tuned model: 0.9510


## Part 3: Random Search — Sample the Space

With many knobs, the grid explodes combinatorially. `RandomizedSearchCV`
samples a fixed budget of combinations from distributions instead.

In [4]:
# WHAT: Run RandomizedSearchCV with an 8-fit budget over continuous ranges.
# WHY: Random search covers wide ranges at fixed cost - and Bergstra & Bengio showed it often matches grid search.

print("Part 3: Random Search")
print("-" * 60)

param_dist = {
    'n_estimators': randint(30, 150),
    'max_depth': randint(3, 15),
    'min_samples_split': randint(2, 10),
}
n_iter = 8
print(f"Distributions: n_estimators ~ [30,150), max_depth ~ [3,15), min_samples_split ~ [2,10)")
print(f"Budget: {n_iter} sampled combinations x {n_folds} folds (+1 refit) = {n_iter * n_folds + 1} fits")

start = time.time()
rand = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                          param_dist, n_iter=n_iter, cv=n_folds,
                          random_state=42, n_jobs=-1)
rand.fit(X_train, y_train)
rand_time = time.time() - start

print(f"\nRandom search finished in {rand_time:.2f}s")
print(f"Best parameters: {rand.best_params_}")
print(f"Best CV accuracy: {rand.best_score_:.4f}")
rand_test = rand.score(X_test, y_test)
print(f"Held-out test accuracy of the tuned model: {rand_test:.4f}")

Part 3: Random Search
------------------------------------------------------------
Distributions: n_estimators ~ [30,150), max_depth ~ [3,15), min_samples_split ~ [2,10)
Budget: 8 sampled combinations x 3 folds (+1 refit) = 25 fits



Random search finished in 0.35s
Best parameters: {'max_depth': 10, 'min_samples_split': 9, 'n_estimators': 32}
Best CV accuracy: 0.9601
Held-out test accuracy of the tuned model: 0.9510


In [5]:
# WHAT: Print the measured comparison table - fits, time, CV and test accuracy per strategy.
# WHY: Every number here was measured in this run - the workflow, not folklore, is the takeaway.

print("=" * 60)
print("Summary: Grid vs Random (measured on THIS machine)")
print("=" * 60)
print(f"""
Data: {X.shape[0]} real breast-cancer biopsies x {X.shape[1]} measurements
      (Wisconsin Diagnostic Breast Cancer, Street et al. 1993)

Strategy         Fits   Time     Best CV acc   Test acc
Baseline (none)   {n_folds}    {baseline_time:5.2f}s   {baseline_score:.4f}        -
Grid search      {n_combos * n_folds + 1:3d}    {grid_time:5.2f}s   {grid.best_score_:.4f}        {grid_test:.4f}
Random search    {n_iter * n_folds + 1:3d}    {rand_time:5.2f}s   {rand.best_score_:.4f}        {rand_test:.4f}

Tuning gain over the untuned baseline:
  Grid search:   {grid.best_score_ - baseline_score:+.4f} CV accuracy for {grid_time / baseline_time:.0f}x the compute
  Random search: {rand.best_score_ - baseline_score:+.4f} CV accuracy for {rand_time / baseline_time:.0f}x the compute

Takeaways (from the numbers above, not from folklore):
  - Both searches explore the cost multiplication from Example 07 first-hand
  - Grid search is exhaustive but its cost grows multiplicatively per knob
  - Random search caps the budget (n_iter) regardless of how many knobs exist
  - On this dataset a default random forest is already close to the ceiling, so
    the tuning gain is small. That is a REAL and common outcome: always compare
    against the untuned baseline before claiming a tuning win.

Next: Example 09 switches to unsupervised learning (no labels at all).
""")
print("✅ Hyperparameter tuning with grid and random search - complete!")

Summary: Grid vs Random (measured on THIS machine)

Data: 569 real breast-cancer biopsies x 30 measurements
      (Wisconsin Diagnostic Breast Cancer, Street et al. 1993)

Strategy         Fits   Time     Best CV acc   Test acc
Baseline (none)   3     0.17s   0.9624        -
Grid search       25     3.19s   0.9624        0.9510
Random search     25     0.35s   0.9601        0.9510

Tuning gain over the untuned baseline:
  Grid search:   +0.0000 CV accuracy for 18x the compute
  Random search: -0.0023 CV accuracy for 2x the compute

Takeaways (from the numbers above, not from folklore):
  - Both searches explore the cost multiplication from Example 07 first-hand
  - Grid search is exhaustive but its cost grows multiplicatively per knob
  - Random search caps the budget (n_iter) regardless of how many knobs exist
  - On this dataset a default random forest is already close to the ceiling, so
    the tuning gain is small. That is a REAL and common outcome: always compare
    against the u

## 📚 References

1. Bergstra, J., & Bengio, Y. (2012). *Random Search for Hyper-Parameter Optimization*. Journal of Machine Learning Research, 13, 281-305. <https://jmlr.org/papers/v13/bergstra12a.html>
2. Snoek, J., Larochelle, H., & Adams, R. P. (2012). *Practical Bayesian Optimization of Machine Learning Algorithms*. NeurIPS 25. <https://arxiv.org/abs/1206.2944>
3. Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework*. KDD 2019. <https://arxiv.org/abs/1907.10902>